In [1]:
from email_classifier.utils import process_email

In [2]:
test_emails = [
    {
        "id"     : 1,
        "subject": "Can't login — paid for annual plan last week",
        "sender" : "cto@bigclient.com",
        "body"   : "Our entire team can't login. We paid for annual "
                   "plan last week. Board demo in 3 hours.",
        "correct": "Billing",
        "model_output": "Billing"
    },
    {
        "id"     : 2,
        "subject": "App crashes on settings page",
        "sender" : "user@startup.com",
        "body"   : "Every time I open Settings the app crashes. "
                   "Chrome 120, Windows 11.",
        "correct": "Technical"
    },
    {
        "id"     : 3,
        "subject": "Evaluating if we stay on your platform",
        "sender" : "ceo@bigclient.com",
        "body"   : "We have been customers for 2 years. Evaluating competitors. "
                   "Zapier missing, bulk export missing, pricing jumped 40%.",
        "correct": "Billing"
    },
]
for email in test_emails:
    result = process_email(email)
    print(f"Email ID: {result['email_id']}")
    print(f"Classification: {result['classification']}")
    print(f"Routed To: {result['routed_to']}")
    print(f"Drafted Response: {result['drafted_response']}")
    print("-" * 50)

Email ID: 1
Classification: {'category': ': Billing', 'urgency': 'High', 'confidence': 66.66666666666666}
Routed To: support@company.com
Drafted Response: None
--------------------------------------------------
Email ID: 2
Classification: {'category': 'Technical', 'urgency': 'High', 'full_output': "THINKING:\n1. Surface complaint: The app crashes when the user opens the settings page.\n2. Root cause: This issue is likely due to a technical problem within the application, possibly related to compatibility with the user's browser or device.\n3. Team that OWNS this: Technical team.\n4. Business impact: This can significantly impact the user experience, as the user cannot access important settings, potentially affecting their ability to utilize the product effectively.\n\nCATEGORY: Technical\nURGENCY: High"}
Routed To: tech-support@company.com
Drafted Response: None
--------------------------------------------------
Email ID: 3
Classification: {'category': ': Feature Request', 'urgency': '

In [2]:
email = [{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 },
 {'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 }]

# results = []

# for i in email:
#     results.append(classify_with_cot(i)['category'])

In [5]:
# print(draft_response_with_tot(email[0],'Technical'))

In [2]:
import json
from dotenv import load_dotenv
from openai import OpenAI
import os
import time
load_dotenv()


client = OpenAI()


def call_llm(prompt):
    response = client.chat.completions.create(
                 model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
               
            )
    return response.choices[0].message.content.strip()

In [9]:
def load_prompt(filename):
    path = os.path.join('prompts',filename)
    with open(path,'r') as f:
        return f.read()

In [38]:
# inside function 
def classify_with_cot(email):
    template = load_prompt('classify_cot.md')
    prompt = template.format(subject= email['subject'],body = email['body'],sender=email['sender'])
    result = call_llm(prompt)

    # extraction logic
    category, urgency = None, None
    for line in result.split('\n'):
        if 'CATEGORY' in line:
            category = line.split(':')[1].strip()
        if 'URGENCY' in line:
            urgency = line.split(':')[1].strip()

    return {
        'category': category,
        'urgency' : urgency,
        'full_output': result
    }
    


In [56]:
from collections import Counter
def classify_with_self_consistency(email,n_runs=5):
    template = load_prompt('classify_cot.md')
    results = []
    traces = []
    for i in range(n_runs):
        prompt = template.format(
            subject=email["subject"],
            body   =email["body"],
            sender =email["sender"]
        )
        result = call_llm(prompt)
        traces.append(result)

        # extract the category ??
        for line in result.split("\n"):
            if "CATEGORY" in line:
                results.append(line.replace("CATEGORY",'').strip())

    votes = Counter(results)
    winner = votes.most_common(1)[0]
    confidance_score = winner[1]/n_runs *100
    urgency = None
    for line in traces[0].split("\n"):      
          if "URGENCY:" in line:
            urgency = line.replace("URGENCY:", "").strip()
    return {
        "category": winner[0],
        "urgency": urgency,
        "confidence": confidance_score,
    }





In [57]:
email = {'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 }

In [58]:
classify_with_self_consistency(email)

{'category': ': Technical', 'urgency': 'Medium', 'confidence': 100.0}

In [41]:
result = ["billing",'technical','billing']
from collections import Counter
test = Counter(result)

In [49]:
test.most_common(1)[0][1]

2

In [39]:
email = {'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 }

classify_with_cot(email)

{'category': 'Technical',
 'urgency': 'High',
 'full_output': "THINKING:\n1. Surface complaint: The app crashes when accessing the settings page.\n2. Root cause: This is likely a technical issue related to the app's compatibility with the browser or a bug in the application.\n3. Team that OWNS this: Technical team (Engineering/Development).\n4. Business impact: High—if the app cannot be used due to continuous crashes, it prevents the user from configuring settings, which could hinder overall productivity and satisfaction.\n\nCATEGORY: Technical\nURGENCY: High"}

In [12]:
template = load_prompt('classify_cot.md')

In [26]:
email = {'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 }

prompt = template.format(subject= email['subject'],body = email['body'],sender=email['sender'])

In [27]:
result = call_llm(prompt)

In [28]:
print(result)

THINKING:
1. Surface complaint: The app crashes when trying to access the settings page.
2. Root cause: This indicates a technical issue with the app functionality on the specified platform (Chrome on Windows 11).
3. Team that OWNS this: Technical support team.
4. Business impact: High, as the user cannot utilize the app's settings, potentially affecting their overall experience and usage of the product.

CATEGORY: Technical
URGENCY: High


In [35]:
category, urgency = None, None
for line in result.split('\n'):
    if 'CATEGORY' in line:
        category = line.split(':')[1].strip()
    if 'URGENCY' in line:
        urgency = line.split(':')[1].strip()

In [36]:
category, urgency

('Technical', 'High')

In [30]:
email['model_output']=result.split('\n')[6].split(':')[1].strip()

In [31]:
email

{'id': 2,
 'subject': 'App crashes on settings page',
 'sender': 'user@bigclient.com',
 'body': 'Every time I open Settings the app crashes. Chrome 120, Windows 11.',
 'correct': 'Technical',
 'model_output': 'Technical'}

In [32]:
result.split('\n')

['THINKING:',
 '1. Surface complaint: The app crashes when trying to access the settings page.',
 '2. Root cause: This indicates a technical issue with the app functionality on the specified platform (Chrome on Windows 11).',
 '3. Team that OWNS this: Technical support team.',
 "4. Business impact: High, as the user cannot utilize the app's settings, potentially affecting their overall experience and usage of the product.",
 '',
 'CATEGORY: Technical',
 'URGENCY: High']